<a href="https://colab.research.google.com/github/kareeennt/DataReto1/blob/main/Reto_01_LIMPIEZA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reto 01: Limpieza de los datos

En este cuaderno preparamos la base INEC 2024. Revisamos los datos necesarios, calculamos los cuatro indicadores y guardamos un CSV limpio para utilizarlo en el notebook de agrupamiento.

In [7]:
import pandas as pd
import numpy as np
from google.colab import files

URL_CSV = "https://raw.githubusercontent.com/kareeennt/DataReto1/main/BDD_GIRS_2024_VF.csv"
df_raw = pd.read_csv(URL_CSV, sep=";", encoding="cp1252", dtype="string", keep_default_na=False)
print("Base original:", df_raw.shape)
display(df_raw.head())

Base original: (221, 213)


,CantonId,NombreCantón,Provin,rgnatu,Pob,base,MR111,MR112,MR113,MR1131,...,PTEX,PLFA,PPIL,PPDPT,POTROS,PTOTAL,MTRO,MTROD,MCTR,MGDSP
0,101,CUENCA,1,1,374870,1,2,2,1,1,...,,,,,,,6,6,1,2
1,102,GIRÓN,1,1,4491,1,2,2,1,2,...,,,,,,,18,1,1,1
2,103,GUALACEO,1,1,14336,1,2,2,1,1,...,,,,,,,18,19,1,2
3,104,NABÓN,1,1,1301,1,2,2,1,2,...,"3,28","0,35","0,15","7,29","7,64",100,18,19,1,
4,105,PAUTE,1,1,9056,1,1,2,2,,...,3,2,3,3,0,100,18,19,2,


## Revisión de las variables

Revisamos las cantidades, los valores vacíos y los ceros de las variables necesarias. Las columnas numéricas se convierten teniendo en cuenta las comas decimales.

In [8]:
candidatas = ['VB', 'CSB', 'CRR', 'TROT', 'RI', 'RIR', 'DIF', 'MR48', 'PORG', 'Pob']
df_cand = df_raw[candidatas].apply(
    lambda s: pd.to_numeric(s.str.replace(',', '.', regex=False), errors='coerce')
)
resumen = pd.DataFrame({
    'Vacíos': df_cand.isna().sum(),
    'Ceros': (df_cand == 0).sum(),
    'Mínimo': df_cand.min(),
    'Mediana': df_cand.median(),
    'Máximo': df_cand.max()
})
display(resumen.round(2))

,Vacíos,Ceros,Mínimo,Mediana,Máximo
VB,0,2,0.0,18.11,4840.0
CSB,5,1,0.0,1.0,2.13
CRR,0,0,15.3,369.3,121905.6
TROT,0,156,0.0,0.0,9939.79
RI,0,134,0.0,0.0,1420069.17
RIR,0,138,0.0,0.0,1420.07
DIF,0,150,0.0,0.0,2000.0
MR48,0,0,0.09,12.0,5586.0
PORG,113,2,0.0,58.96,86.43
Pob,0,0,507.0,9032.0,2853214.0


## Comprobación de los casos especiales

Revisamos las respuestas del formulario para diferenciar ceros de datos faltantes. También comprobamos los municipios con cobertura de barrido vacía o mayor al 100 %.

In [9]:
# Preguntas filtro del formulario (1 = Sí, 2 = No) y variables de apoyo
tipos_rec = ['MR43411', 'MR43412', 'MR43413', 'MR43414', 'MR43415', 'MR43416', 'MR43417',
             'MR43418', 'MR43419', 'MR434110', 'MR434111', 'MR434112', 'MR434113']
cols_filtro = tipos_rec + ['MR441', 'MR442', 'MR452', 'MR421', 'MR4223']
df_filtro = df_raw[cols_filtro].apply(lambda s: pd.to_numeric(s.str.replace(',', '.'), errors='coerce'))

# RI: ¿los ceros coinciden con municipios que no declaran recuperar ningún tipo de residuo?
declara_rec = (df_filtro[tipos_rec] == 1).any(axis=1)
print("RI > 0 según si declara recuperar algún residuo:")
display(pd.crosstab(declara_rec.rename('declara recuperar'), (df_cand['RI'] > 0).rename('RI > 0')))

# TROT: ¿los ceros coinciden con municipios que no aprovechan orgánicos (MR441 mercados, MR442 ciudad)?
print("TROT > 0 según MR441 y MR442:")
display(pd.crosstab([df_filtro['MR441'], df_filtro['MR442']], (df_cand['TROT'] > 0).rename('TROT > 0')))

# DIF: ¿los ceros coinciden con municipios sin recolección diferenciada (MR452)?
print("DIF > 0 según MR452:")
display(pd.crosstab(df_filtro['MR452'], (df_cand['DIF'] > 0).rename('DIF > 0')))

# CSB: casos faltantes o mayores a 1, con la pregunta de servicio de barrido (MR421)
# y los km susceptibles de barrido (MR4223)
mask = df_cand['CSB'].isna() | (df_cand['CSB'] > 1)
display(pd.concat([df_raw['NombreCantón'], df_cand[['CSB', 'VB']], df_filtro[['MR421', 'MR4223']]], axis=1)[mask])

RI > 0 según si declara recuperar algún residuo:


RI > 0,False,True
declara recuperar,,
False,123,0
True,11,87


TROT > 0 según MR441 y MR442:


TROT > 0     False  True 
MR441 MR442              
1     1          0     28
      2          1     18
2     1          0     16
      2        155      3

DIF > 0 según MR452:


DIF > 0,False,True
MR452,,
1,0,71
2,150,0


,NombreCantón,CSB,VB,MR421,MR4223
23,BIBLIÁN,1.17,70.0,1,60.0
28,SUSCAL,1.04,50.0,1,48.0
35,LATACUNGA,1.46,40.16,1,27.46
141,MONTECRISTI,<NA>,70.06,1,0.0
144,ROCAFUERTE,<NA>,19.93,1,0.0
188,PATATE,<NA>,18.06,1,0.0
189,QUERO,<NA>,20.0,1,0.0
201,PAQUISHA,<NA>,0.0,2,0.0
203,ISABELA,2.13,5.75,1,2.7


## Cálculo de los indicadores

Calculamos la cobertura de barrido, la proporción de inorgánicos recuperados, la proporción de recolección diferenciada y el logaritmo de residuos recolectados. Convertimos los kilogramos a toneladas y aproximamos los meses a 30 días para que las unidades sean compatibles.

In [10]:
# Matriz con los 4 indicadores elegidos (la base df_cand sale de la celda de variables candidatas)
base = df_raw[['CantonId', 'NombreCantón']].copy()
base['CSB'] = df_cand['CSB']
base['p_inorg'] = df_cand['RI'] / 1000 / df_cand['CRR']   # RI pasa de kg/mes a ton/mes
base['p_dif'] = df_cand['DIF'] * 30 / df_cand['CRR']      # DIF pasa de ton/día a ton/mes
base['log_CRR'] = np.log(df_cand['CRR'])                  # tamaño, en escala logarítmica

# Paquisha no tiene servicio de barrido: cobertura 0 por definición, no dato faltante
base.loc[base['NombreCantón'] == 'PAQUISHA', 'CSB'] = 0

# Registros que quedan y municipios que se excluyen
indicadores = ['CSB', 'p_inorg', 'p_dif', 'log_CRR']
print("Registros completos:", base[indicadores].dropna().shape[0])
display(base[base[indicadores].isna().any(axis=1)][['CantonId', 'NombreCantón']])

Registros completos: 217


,CantonId,NombreCantón
141,1309,MONTECRISTI
144,1312,ROCAFUERTE
188,1805,PATATE
189,1806,QUERO


## Selección de registros

Excluimos los municipios donde no se puede calcular la cobertura. Conservamos a Paquisha con cobertura cero, los ceros reportados y los valores atípicos para no cambiar los datos sin una justificación.

In [11]:
# Municipios que no se pueden usar: la cobertura de barrido (CSB) no se puede calcular
excluidos = base[base[indicadores].isna().any(axis=1)][['CantonId', 'NombreCantón']].copy()

# Evidencia tomada del formulario: MR421 = 1 (barren) y MR4223 = 0 (km susceptibles de barrido)
excluidos = excluidos.join(df_filtro[['MR421', 'MR4223']])
excluidos['motivo'] = 'Barre (MR421 = 1) pero declara 0 km susceptibles (MR4223 = 0): CSB no se puede calcular'
display(excluidos)

# Matriz de trabajo: se mantiene el índice original para poder volver a df_raw
df_matriz = base.drop(index=excluidos.index)
print("Registros conservados:", df_matriz.shape[0])
print("Faltantes en los indicadores:", df_matriz[indicadores].isna().sum().sum())

,CantonId,NombreCantón,MR421,MR4223,motivo
141,1309,MONTECRISTI,1,0.0,Barre (MR421 = 1) pero declara 0 km susceptibl...
144,1312,ROCAFUERTE,1,0.0,Barre (MR421 = 1) pero declara 0 km susceptibl...
188,1805,PATATE,1,0.0,Barre (MR421 = 1) pero declara 0 km susceptibl...
189,1806,QUERO,1,0.0,Barre (MR421 = 1) pero declara 0 km susceptibl...


Registros conservados: 217
Faltantes en los indicadores: 0


## Guardar el CSV limpio

Guardamos las columnas originales de los 217 municipios conservados y los cuatro indicadores calculados. También guardamos el índice original para que el otro notebook pueda relacionar correctamente los datos con los municipios.

In [12]:
# Conservamos las columnas originales y añadimos los indicadores preparados.
df_limpio = df_raw.loc[df_matriz.index].copy()
df_limpio.insert(0, '_indice_original', df_limpio.index)
df_limpio['_CSB_modelo'] = df_matriz['CSB'].to_numpy()
for variable in ['p_inorg', 'p_dif', 'log_CRR']:
    df_limpio[variable] = df_matriz[variable].to_numpy()

assert len(df_limpio) == 217, 'Revisar los registros conservados'
assert np.isfinite(df_matriz[indicadores].to_numpy(dtype=float)).all()

nombre_archivo = 'BDD_GIRS_2024_LIMPIO.csv'
df_limpio.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
print('CSV generado:', nombre_archivo)
print('Dimensiones:', df_limpio.shape)
display(df_limpio[['_indice_original', 'CantonId', 'NombreCantón', '_CSB_modelo', 'p_inorg', 'p_dif', 'log_CRR']].head())
files.download(nombre_archivo)

CSV generado: BDD_GIRS_2024_LIMPIO.csv
Dimensiones: (217, 218)


,_indice_original,CantonId,NombreCantón,_CSB_modelo,p_inorg,p_dif,log_CRR
0,0,101,CUENCA,0.86,0.009443,0.0,9.555928
1,1,102,GIRÓN,0.28,0.018045,1.0,4.859812
2,2,103,GUALACEO,1.00,0.000000,0.0,6.173786
3,3,104,NABÓN,1.00,0.000000,0.0,4.317488
4,4,105,PAUTE,1.00,0.000128,0.0,5.886104


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>